In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options as ChromeOptions
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC

import requests
import pandas as pd
from datetime import date, timedelta
from functools import reduce
import os
from shutil import which
from bs4 import BeautifulSoup

In [3]:
def _setup_chrome_options():
    """Set up Chrome options for local Selenium execution."""
    options = ChromeOptions()
    
    # Keep Chrome visible while running locally.
    # Uncomment this when you want headless execution again.
    # options.add_argument("--headless=new")
    
    options.add_argument("--start-maximized")
    
    return options

In [6]:
url = "https://www.rotowire.com/hockey/nhl-lineups.php"

options = _setup_chrome_options()
driver = webdriver.Chrome(options=options)

driver.set_page_load_timeout(20)
driver.set_script_timeout(20)

driver.get(url)

print("Page title:", driver.title)
print("Current URL:", driver.current_url)

Page title: NHL Starting Lineups: Hockey Daily Lineups | RotoWire
Current URL: https://www.rotowire.com/hockey/nhl-lineups.php


In [7]:
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CLASS_NAME, "lineups"))
)

print("Lineup container found!")

Lineup container found!


In [8]:
matchups = []

lineups = (
    driver.find_element(By.CLASS_NAME, "lineups")
    .find_elements(By.CLASS_NAME, "is-nhl")
)

print("Number of lineups found:", len(lineups))

Number of lineups found: 14


In [9]:
for lineup in lineups[:-1]:
    goalies = lineup.find_elements(
        By.CLASS_NAME,
        "lineup__player-highlight-name"
    )

    away_goalie = goalies[0].text.strip().split()[-1]
    home_goalie = goalies[1].text.strip().split()[-1]

    teams = lineup.find_element(
        By.CLASS_NAME,
        "lineup__matchup"
    ).find_elements(By.TAG_NAME, "a")

    if len(teams) >= 2:
        away_team = teams[0].text.split(" (")[0]
        home_team = teams[1].text.split(" (")[0]

        matchups.append(
            (away_team, home_team, away_goalie, home_goalie)
        )

matchups

[('Blackhawks', 'Sabres', 'Knight', 'Luukkonen'),
 ('Canadiens', 'Penguins', 'Dobes', 'Silovs'),
 ('Capitals', 'Lightning', 'Lindgren', 'Vasilevskiy'),
 ('Senators', 'Maple Leafs', 'Ullmark', 'Bobrovsky'),
 ('Kraken', 'Oilers', 'Daccord', 'Jarry'),
 ('Hurricanes', 'Flyers', 'Kochetkov', 'Woll'),
 ('Mammoth', 'Blue Jackets', 'Vejmelka', 'Greaves'),
 ('Devils', 'Islanders', 'Daws', 'Sorokin'),
 ('Bruins', 'Wild', 'DiPietro', 'Wallstedt'),
 ('Stars', 'Predators', 'DeSmith', 'Saros'),
 ('Blues', 'Avalanche', 'Binnington', 'Blackwood'),
 ('Flames', 'Canucks', 'Wolf', 'Merilainen'),
 ('Kings', 'Sharks', 'Kuemper', 'Askarov')]

In [10]:
driver.quit()

In [11]:
url = "https://moneypuck.com/goalies.htm"

options = _setup_chrome_options()
driver = webdriver.Chrome(options=options)

driver.set_page_load_timeout(20)
driver.set_script_timeout(20)

driver.get(url)

print("Page title:", driver.title)
print("Current URL:", driver.current_url)

Page title: NHL Goalie Advanced Stats 2026-2027
Current URL: https://moneypuck.com/goalies.htm


In [12]:
wait = WebDriverWait(driver, 10)

dropdown = wait.until(
    EC.presence_of_element_located((By.ID, "num_games_type"))
)

select = Select(dropdown)

select.select_by_value("_20")

print("Selected Last 20 Games")

Selected Last 20 Games


In [13]:
wait = WebDriverWait(driver, 10)

dropdown = wait.until(
    EC.presence_of_element_located((By.ID, "num_games_type"))
)

select = Select(dropdown)

select.select_by_value("_20")

print("Selected Last 20 Games")

Selected Last 20 Games


In [14]:
wait.until(
    EC.presence_of_element_located((By.ID, "goaliesTable"))
)

print("Goalie table found!")

Goalie table found!


In [15]:
table_html = driver.find_element(
    By.ID,
    "goaliesTable"
).get_attribute("outerHTML")

print("HTML length:", len(table_html))

HTML length: 83484


In [16]:
driver.quit()

In [17]:
soup = BeautifulSoup(table_html, "html.parser")

rows = soup.find_all("tr")[1:]

data = []

for tr in rows:
    tds = tr.find_all("td")

    if len(tds) < 7:
        continue

    try:
        rank = int(tds[0].get_text(strip=True))
        goalie = tds[2].get_text(strip=True).split()[-1]
        gsa = float(tds[6].get_text(strip=True))
    except:
        continue

    data.append({
        "Rank": rank,
        "Goalie": goalie,
        "GSAx": gsa
    })

goalie_df = pd.DataFrame(data)

goalie_df.head(20)

,Rank,Goalie,GSAx
0,1,Hofer,2.9
1,2,Swayman,2.5
2,3,Markstrom,2.5
3,4,Shesterkin,2.3
4,5,Thompson,2.0
5,6,Vejmelka,1.7
6,7,Garand,1.7
7,8,Silovs,1.6
8,9,Stolarz,1.6
9,10,Daccord,1.5


In [18]:
goalie_ranks = dict(
    zip(goalie_df["Goalie"], goalie_df["Rank"])
)

goalie_ranks

{'Hofer': 1,
 'Swayman': 2,
 'Markstrom': 3,
 'Shesterkin': 4,
 'Thompson': 5,
 'Vejmelka': 6,
 'Garand': 7,
 'Silovs': 8,
 'Stolarz': 9,
 'Daccord': 10,
 'Woll': 11,
 'Gibson': 12,
 'Askarov': 13,
 'Dobes': 14,
 'Sorokin': 15,
 'Wallstedt': 16,
 'Oettinger': 17,
 'Dostal': 18,
 'Merilinen': 19,
 'Saros': 20,
 'Greaves': 21,
 'Allen': 22,
 'Schmid': 23,
 'Knight': 24,
 'Bobrovsky': 25,
 'Skinner': 26,
 'Bussi': 27,
 'Blackwood': 28,
 'Hart': 29,
 'Vasilevskiy': 30,
 'Soderblom': 31,
 'Wolf': 32,
 'Vladar': 33,
 'Luukkonen': 34,
 'Jarry': 35,
 'Levi': 36,
 'Kuemper': 37,
 'Lankinen': 38}

In [19]:
url = "https://www.nextteamup.com/nhl"

headers = {
    "User-Agent": "Mozilla/5.0"
}

resp = requests.get(
    url,
    headers=headers,
    timeout=15
)

resp.raise_for_status()

print("Status:", resp.status_code)
print("Response length:", len(resp.text))

Status: 200
Response length: 59315


In [20]:
soup = BeautifulSoup(resp.text, "html.parser")

streaks = {}

rows = soup.select("table tbody tr")

print("Rows found:", len(rows))

Rows found: 32


In [21]:
soup = BeautifulSoup(resp.text, "html.parser")

streaks = {}

rows = soup.select("table tbody tr")

print("Rows found:", len(rows))

Rows found: 32


In [22]:
today = date.today()
start = today - timedelta(days=30)

url = (
    "https://www.naturalstattrick.com/teamtable.php"
    f"?fromseason=20252026"
    f"&thruseason=20252026"
    f"&stype=2"
    f"&sit=5v5"
    f"&score=all"
    f"&rate=n"
    f"&team=all"
    f"&loc=B"
    f"&gpf=410"
    f"&fd={start}"
    f"&td={today}"
)

print(url)

https://www.naturalstattrick.com/teamtable.php?fromseason=20252026&thruseason=20252026&stype=2&sit=5v5&score=all&rate=n&team=all&loc=B&gpf=410&fd=2026-09-03&td=2026-10-03


In [ ]:
today = date.today()
start = today - timedelta(days=30)

url = (
    "https://www.naturalstattrick.com/teamtable.php"
    f"?fromseason=20262027"
    f"&thruseason=20262027"
    f"&stype=2"
    f"&sit=5v5"
    f"&score=all"
    f"&rate=n"
    f"&team=all"
    f"&loc=B"
    f"&gpf=410"
    f"&fd={start}"
    f"&td={today}"
)

print(url)

https://www.naturalstattrick.com/teamtable.php?fromseason=20252026&thruseason=20252026&stype=2&sit=5v5&score=all&rate=n&team=all&loc=B&gpf=410&fd=2026-09-03&td=2026-10-03


In [25]:
options = _setup_chrome_options()
driver = webdriver.Chrome(options=options)

driver.set_page_load_timeout(30)
driver.set_script_timeout(30)

driver.get(url)

print("Page title:", driver.title)
print("Current URL:", driver.current_url)

Page title: Just a moment...
Current URL: https://www.naturalstattrick.com/teamtable.php?fromseason=20252026&thruseason=20252026&stype=2&sit=5v5&score=all&rate=n&team=all&loc=B&gpf=410&fd=2026-09-03&td=2026-10-03


In [26]:
html = driver.page_source

print("Page title:", driver.title)
print("Current URL:", driver.current_url)
print("HTML length:", len(html))
print("Contains table:", "<table" in html)
print("Contains Cloudflare:", "Cloudflare" in html)

Page title: Team Season Totals - Natural Stat Trick
Current URL: https://www.naturalstattrick.com/teamtable.php?fromseason=20252026&thruseason=20252026&stype=2&sit=5v5&score=all&rate=n&team=all&loc=B&gpf=410&fd=2026-09-03&td=2026-10-03
HTML length: 220986
Contains table: True
Contains Cloudflare: False


In [27]:
df = pd.read_html(
    html,
    flavor="lxml",
    header=0,
    index_col=0,
    na_values=["-"]
)[0]

df.head()

,Team,GP,TOI,W,L,OTL,ROW,Points,Point %,CF,...,LDSA,LDSF%,LDGF,LDGA,LDGF%,LDSH%,LDSV%,SH%,SV%,PDO
1,Chicago Blackhawks,82,4006:50,29,39,14,26,72,0.439,3346,...,792,46.49,16,27,37.21,2.33,96.59,9.56,90.00,0.996
2,Florida Panthers,82,3877:00,40,38,4,35,84,0.512,3755,...,714,51.26,22,29,43.14,2.93,95.94,9.62,89.30,0.989
3,Pittsburgh Penguins,82,4010:59,41,25,16,38,98,0.598,3876,...,721,48.90,33,31,51.56,4.78,95.70,11.13,90.10,1.012
4,New York Rangers,82,4035:07,34,39,9,31,77,0.470,3679,...,816,45.74,26,28,48.15,3.78,96.57,9.56,91.33,1.009
5,Colorado Avalanche,82,3938:26,55,16,11,51,121,0.738,4612,...,744,55.98,47,27,63.51,4.97,96.37,9.84,92.20,1.020


In [ ]:
cf = df.sort_values(by="CF%", ascending=False, ignore_index=True)
cf.index += 1
cf.reset_index(inplace=True)
cf = cf.rename(columns={"index": "CF%"})
cf = cf[["Team", "CF%"]]

gf = df.sort_values(by="GF%", ascending=False, ignore_index=True)
gf.index += 1
gf.reset_index(inplace=True)
gf = gf.rename(columns={"index": "GF%"})
gf = gf[["Team", "GF%"]]

xgf = df.sort_values(by="xGF%", ascending=False, ignore_index=True)
xgf.index += 1
xgf.reset_index(inplace=True)
xgf = xgf.rename(columns={"index": "xGF%"})
xgf = xgf[["Team", "xGF%"]]

hdcf = df.sort_values(by="HDCF%", ascending=False, ignore_index=True)
hdcf.index += 1
hdcf.reset_index(inplace=True)
hdcf = hdcf.rename(columns={"index": "HDCF%"})
hdcf = hdcf[["Team", "HDCF%"]]

sh = df.sort_values(by="SH%", ascending=False, ignore_index=True)
sh.index += 1
sh.reset_index(inplace=True)
sh = sh.rename(columns={"index": "SH%"})
sh = sh[["Team", "SH%"]]

In [ ]:
cf.head()

In [ ]:
dfs = [cf, gf, xgf, hdcf, sh]

final_df = reduce(
    lambda left, right: pd.merge(
        left,
        right,
        on="Team",
        how="outer"
    ),
    dfs
)

final_df

In [ ]:
def normalize_team(name):
    return (
        name.lower()
        .replace(".", "")
        .replace("é", "e")
        .replace(" ", "")
    )

In [ ]:
normalized_streaks = {
    normalize_team(team): streak
    for team, streak in streaks.items()
}

normalized_streaks

In [ ]:
res = pd.DataFrame()

for away, home, away_goalie, home_goalie in matchups:

    away_df = final_df[
        final_df["Team"].str.contains(
            away,
            case=False,
            na=False
        )
    ]

    home_df = final_df[
        final_df["Team"].str.contains(
            home,
            case=False,
            na=False
        )
    ]

    away_rank = goalie_ranks.get(away_goalie)
    home_rank = goalie_ranks.get(home_goalie)

    away_df = away_df.assign(
        Goalie=away_goalie,
        Goalie_Rank=away_rank,
        Streak=away_df["Team"].apply(
            lambda t: normalized_streaks.get(
                normalize_team(t)
            )
        )
    )

    home_df = home_df.assign(
        Goalie=home_goalie,
        Goalie_Rank=home_rank,
        Streak=home_df["Team"].apply(
            lambda t: normalized_streaks.get(
                normalize_team(t)
            )
        )
    )

    matchup_df = pd.concat(
        [away_df, home_df],
        ignore_index=True
    )

    res = pd.concat(
        [res, matchup_df],
        ignore_index=True
    )

In [ ]:
output_file = "../public/result.csv"

res.to_csv(output_file, index=False)

print(f"Saved {len(res)} rows to {output_file}")